# Featuresmith Tutorial: 05 — Comparing Dataset Versions with Dataset Diff Engine

Learn how to compare dataset snapshot versions with `fs.diff()`, `fs.diff_findings()`, and `fs.render_diff()` to prevent silent schema drift, missingness spikes, and quality regressions.

---


## 1. Why Dataset Diffing Matters
In production ML pipelines, datasets evolve continuously. New snapshots arrive daily or weekly. Silent changes — such as dropped columns, renamed features, type shifts, or missing value spikes — can break model inference or corrupt retrained models.

Featuresmith's `fs.diff()` compares two dataset snapshots deterministically and provides an overall health verdict:
- **`unchanged`**: No material structural or quality changes.
- **`improved`**: Quality metrics improved (e.g., missingness decreased, leakage eliminated).
- **`regressed`**: Quality degraded (e.g., columns dropped, missingness spiked, schema broke).

### Prerequisite: Prepare the Sales Dataset
This notebook loads `examples/data/processed/sales.csv`, which `examples/prepare_datasets.py` generates deterministically (no network). From the repository root, run:

```bash
python examples/prepare_datasets.py
```


### Step 1: Simulate Dataset Evolution (Snapshot v1 vs Snapshot v2)

In [1]:
import os

import pandas as pd

import featuresmith as fs

data_path = os.path.join("..", "data", "processed", "sales.csv")
v1 = pd.read_csv(data_path)

v2 = v1.copy()
v2.drop(columns=["store_version"], inplace=True)
v2["promo_code"] = "SUMMER2026"
v2.loc[:100, "discount"] = None

print(f"Snapshot v1 Shape: {v1.shape}")
print(f"Snapshot v2 Shape: {v2.shape}")

Snapshot v1 Shape: (1000, 10)
Snapshot v2 Shape: (1000, 10)


### Step 2: Execute Dataset Diff Engine (`fs.diff`)

In [2]:
diff_result = fs.diff(v1, v2)

print(f"Health Verdict : {diff_result.summary.overall_health.upper()}")
print(f"Recommendation : {diff_result.summary.recommendation}")
print(f"Added Columns  : {diff_result.schema.added_columns}")
print(f"Removed Columns: {diff_result.schema.removed_columns}")
print(f"Missingness Shift Count: {diff_result.summary.missing_values_increased}")

Health Verdict : REGRESSED
Recommendation : Dataset regressed: 1 column(s) removed; missingness increased in 1 column(s). Review the changes before retraining.
Added Columns  : ('promo_code',)
Removed Columns: ('store_version',)
Missingness Shift Count: 1


### Step 3: Extract Diff Findings & Render Diff Text Report
Use `fs.diff_findings()` to extract `RuleFinding` objects from a diff result, and `fs.render_diff()` for terminal output.

In [3]:
findings = fs.diff_findings(diff_result)
print(f"Derived Diff Findings Count: {len(findings)}")
for f in findings[:3]:
    print(f"  - [{f.severity.upper()}] {f.title}")

diff_report = fs.render_diff(diff_result, target="console")
print("\n=== Formatted Diff Text Report Preview ===")
print(diff_report[:500] + "\n...")

Derived Diff Findings Count: 3
  - [INFO] Columns added to the dataset
  - [WARNING] Columns removed from the dataset
  - [WARNING] Missing values increased in column 'discount'

=== Formatted Diff Text Report Preview ===
Featuresmith Dataset Diff
Rows: 1,000 -> 1,000 (+0 / -0) | Columns: 10 -> 10
Engine: v0.2.0

Rows 0 removed, 0 added; columns 1 removed, 1 added; overall health: regressed.

Dataset Comparison Summary
  Rows Added: 0
  Rows Removed: 0
  Columns Added: 1
  Columns Removed: 1
  Columns Renamed: 0
  Schema Changed: Yes
  Type Changes: 0
  Missing Values Increased: 1 column(s)
  Missing Values Improved: 0 column(s)
  Duplicate Rows Increased: No
  Duplicate Rows Improved: No
  Newly Constant Columns
...


### Key Takeaways & Connection to Next Tutorial
- `fs.diff()` gives an immediate pass/fail verdict for dataset snapshot updates.
- It tracks structural, schema, quality, distribution, and leakage deltas in one canonical object.
- `fs.diff_findings()` converts diff deltas into standard `RuleFinding` objects for CI exit code gating.

**Next Tutorial**: In `06_end_to_end_workflow.ipynb`, we build a production pre-training pipeline gate that integrates review, scoring, and error handling.